# Relatório: Predição do Valor de Imóveis (MEDV)

## 1.Introdução

Este relatório apresenta a aplicação de quatro metodologias de aprendizado de máquina para predizer o valor médio de casas ocupadas pelo proprietário **(medv)** utilizando o conjunto de dados *Boston Housing*.

As metodologias aplicadas são:
1. Regressão Linear
2. Árvores de Regressão
3. Bagging
4. Random Forest

A avaliação dos modelos foi realizada por meio de **validação cruzada Leave-One-Out (LOOCV)**, utilizando como métricas o **Erro Quadrático Médio (MSE)**, o **RMSE**, o **MAE**, o **Coeficiente de Determinação ($R^2$)** e a **Correlação** entre os valores preditos e observados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr

# Configurações estéticas
sns.set_theme(style="whitegrid")

## 2. Carregamento e Preparação dos Dados

In [ ]:
# Definindo nomes das colunas conforme padrão do dataset Boston Housing
column_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']

# Carregando o dataset
data = pd.read_csv('housing.csv', header=None, delimiter=r"\s+", names=column_names)

# Visualizando as primeiras linhas
data.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [ ]:
X = data.drop('MEDV', axis=1)
y = data['MEDV']

loo = LeaveOneOut()

def evaluate_model(model, X, y):
    # cross_val_predict com LOOCV
    y_pred = cross_val_predict(model, X, y, cv=loo)
    
    mse = mean_squared_error(y, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    corr, _ = pearsonr(y, y_pred)
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'Corr': corr,
        'y_pred': y_pred
    }

## 3. Metodologia de Avaliação

Para cada modelo, realizaremos a validação cruzada **Leave-One-Out (LOOCV)**. Nesta técnica, o modelo é treinado $n$ vezes (onde $n$ é o número de observações), deixando uma observação fora para teste em cada iteração.

## 4. Modelos

## 4.1. Regressão Linear

In [5]:
lr = LinearRegression()
res_lr = evaluate_model(lr, X, y)
print(f"Regressão Linear concluída.")

Regressão Linear concluída.


## 4.2. Árvores de Regressão

In [6]:
tree = DecisionTreeRegressor(random_state=1)
res_tree = evaluate_model(tree, X, y)
print(f"Árvore de Regressão concluída.")

Árvore de Regressão concluída.


## 4.3. Bagging

In [7]:
# Padronizado com 100 estimadores e random_state=1
bagging = BaggingRegressor(n_estimators=100, random_state=1)
res_bag = evaluate_model(bagging, X, y)
print(f"Bagging concluído.")

Bagging concluído.


## 4.4. Random Forest

In [8]:
# Padronizado com 100 estimadores e random_state=1
rf = RandomForestRegressor(n_estimators=100, random_state=1)
res_rf = evaluate_model(rf, X, y)
print(f"Random Forest concluído.")

Random Forest concluído.


## 5.Comparação dos Resultados

In [9]:
metrics = [res_lr, res_tree, res_bag, res_rf]
model_names = ['Regressão Linear', 'Árvore de Regressão', 'Bagging', 'Random Forest']

results = pd.DataFrame({
    'Modelo': model_names,
    'MSE': [m['MSE'] for m in metrics],
    'RMSE': [m['RMSE'] for m in metrics],
    'MAE': [m['MAE'] for m in metrics],
    'R2': [m['R2'] for m in metrics],
    'Correlação': [m['Corr'] for m in metrics]
})

# Aplicando estilização avançada
styled_results = results.sort_values(by='RMSE').style \
    .format({"MSE": "{:.2f}", "RMSE": "{:.2f}", "MAE": "{:.2f}", "R2": "{:.4f}", "Correlação": "{:.4f}"}) \
    .background_gradient(cmap='RdYlGn_r', subset=['MSE', 'RMSE', 'MAE']) \
    .background_gradient(cmap='RdYlGn', subset=['R2', 'Correlação']) \
    .set_caption("Comparativo de Performance dos Modelos (LOOCV)") \
    .set_table_styles([
        {'selector': 'caption', 'props': [('color', '#4f4f4f'), ('font-size', '16px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('color', 'black'), ('border', '1px solid #ccc')]} 
    ])

styled_results

,Modelo,MSE,RMSE,MAE,R2,Correlação
3,Random Forest,10.23,3.20,2.18,0.8788,0.9377
2,Bagging,10.44,3.23,2.20,0.8763,0.9363
1,Árvore de Regressão,21.05,4.59,2.87,0.7507,0.8765
0,Regressão Linear,23.73,4.87,3.38,0.7190,0.8480


## 6.Discussão e Conclusão



**6.1. Introdução** 


O conjunto de dados de Boston é um dos conjuntos mais conhecidos do ambito estatístico. Logo de início pode se perceber que se trata de um conjunto com **506 observações** e **14 variáveis**, coletado pelo U.S. Census Service na década de 1970. 

O objetivo original é prever o valor mediano de casas ocupadas por proprietários `(MEDV)` em milhares de dólares. Já as variáveis explicativas incluem taxa de criminalidade `(CRIM)`, proporção de zonas residenciais `(ZN)`, concentração de óxidos nítricos `(NOX)`, número médio de cômodos (RM), entre outras.


**6.2. Comparativo de Performance (**Estimativa LOOCV**)**

Ao executar a validação cruzada Leave-One-Out, os resultados típicos para este dataset seguem esta hierarquia de performance:

| Modelo         | MSE  | RMSE| MAE |    R2 | Correlação|
|-----           | ---- | --- | ----| ------| --------- |
| Random Forest  | 10.23| 3.20| 2.18| 0.8788| 0.9377    |
| Bagging        | 10.44| 3.23| 2.20| 0.8763| 0.9363    |
| Árv. de Regr.  | 21.05| 4.59| 2.87| 0.7507| 0.8765    |
| Regressão Lin. | 23.73| 4.87| 3.38| 0.7190| 0.8480    |

**6.3. Discussão dos Resultados**

**Regressão Linear vs. Métodos Baseados em Árvores**
A Regressão Linear apresentou o maior erro (MSE). Isso ocorre porque ela assume uma relação linear entre as variáveis (como número de quartos e preço), o que nem sempre é capturado na complexidade do mercado imobiliário. Já a Árvore de Regressão conseguiu capturar relações não-lineares e interações entre variáveis, superando a regressão linear mesmo sendo um modelo simples.

**O Poder dos Ensembles (Bagging e Random Forest)**
Houve um salto significativo de performance ao passar da Árvore única para o Bagging e a Random Forest.

* **Bagging**: Ao treinar 100 árvores em amostras diferentes e tirar a média, o modelo reduziu drasticamente a variância (o erro de "instabilidade"). A árvore única tende a sofrer muito com pequenas mudanças nos dados; o Bagging suaviza isso.
* **Random Forest**: Foi o modelo vencedor. Além de fazer o que o Bagging faz, a Random Forest sorteia variáveis em cada corte. Isso "descorrela" as árvores (torna-as diferentes umas das outras), o que melhora a capacidade de generalização e resulta no menor erro quadrático médio.

**6.4. Conclusão**
O desafio demonstra que para problemas de predição de valores imobiliários (*regressão*), métodos de conjunto (*ensembles*) como a **Random Forest** são superiores. O uso da validação cruzada *LOOCV* foi fundamental para garantir que as métricas de erro fossem confiáveis, pois o modelo foi testado em todas as observações do dataset, uma por uma, eliminando o viés de uma divisão simples de treino/teste.

A forte correlação (> 0.90) observada nos modelos de ensemble indica que os valores preditos acompanham muito de perto a tendência real dos preços das casas